# Identifying X-ray bright clusters/groups contaminating LoVoCCS cluster emission

This sub-section of the project....

## Import Statements

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
from astropy.units import Quantity, UnitConversionError
from astropy.cosmology import LambdaCDM
from shutil import rmtree
import os

# This adds the directory above to the path, allowing me to import the common functions that I've written in
#  common.py - this just saves me repeating boring code and makes sure its all consistent
import sys
sys.path.insert(0, '..')
from common import lovoccs_cosmo, haversine

import xga
xga.NUM_CORES = 10
temp_dir = xga.OUTPUT
actual_dir = temp_dir.split('notebooks/')[0]+'notebooks/xga_output/'
xga.OUTPUT = actual_dir
xga.utils.OUTPUT = actual_dir
# As currently XGA will setup an xga_output directory in our current directory, I remove it to keep it all clean
if os.path.exists('xga_output'):
    rmtree('xga_output')
from xga.samples import ClusterSample
from xga.sourcetools import ang_to_rad

# We will ignore some divide by invalid value warnings here - I promise its fine
np.seterr(divide='ignore', invalid='ignore')

%matplotlib inline

## Setting up necessary directories

Here we ensure that the directories we need to store the outputs in have been created:

In [2]:
# fix_ap_dist_path = "../../outputs/figures/positions_and_morphology/centroid_posang/fixed_aperture_centroid_posang_dists/"
# os.makedirs(fix_ap_dist_path, exist_ok=True)

# coords_morph_out_path = "../../outputs/result_files/positions_and_morphology/"
# os.makedirs(coords_morph_out_path, exist_ok=True)

# centroid_posang_vis_path = "../../outputs/cluster_visualisations/centroid_posangle_meas/"
# os.makedirs(centroid_posang_vis_path, exist_ok=True)

## Defining any useful functions

## Loading data files

Important considerations for this dataset:
* 

### X-LoVoCCS-I base sample

In [3]:
xlovoccs_base = pd.read_csv("../../sample_files/X-LoVoCCSI.csv")
xlovoccs_base.insert(0, 'name', xlovoccs_base['LoVoCCSID'].apply(lambda x: "LoVoCCS-" + str(x)))
xlovoccs_base.head(6)

,name,LoVoCCSID,parent_LoVoCCSID,Name,start_ra,start_dec,MCXC_Redshift,MCXC_R500,MCXC_RA,MCXC_DEC,manual_xray_ra,manual_xray_dec,MCXC_Lx500_0.1_2.4
0,LoVoCCS-1,1,1,A2029,227.734300,5.745471,0.0766,1.3344,227.73000,5.720000,227.734300,5.745471,8.726709e+44
1,LoVoCCS-2,2,2,A401,44.740000,13.580000,0.0739,1.2421,44.74000,13.580000,NaN,NaN,6.088643e+44
2,LoVoCCS-4A,4A,4,A85North,10.458750,-9.301944,0.0555,1.2103,10.45875,-9.301944,NaN,NaN,5.100085e+44
3,LoVoCCS-4B,4B,4,A85South,10.451487,-9.460007,0.0555,1.2103,10.45875,-9.301944,10.451487,-9.460007,5.100085e+44
4,LoVoCCS-5,5,5,A3667,303.157313,-56.845978,0.0556,1.1990,303.13000,-56.830000,303.157313,-56.845978,4.871933e+44
5,LoVoCCS-7,7,7,A3827,330.480000,-59.950000,0.0980,1.1367,330.48000,-59.950000,NaN,NaN,4.204419e+44


### X-LoVoCCS-I ICM peak positions

As we wish to calculate the centroid shift as part of this analysis, we also read in the previously-measured X-ray peak coordinates for the LoVoCCS sample:

In [4]:
xlovoccs_peaks = pd.read_csv(coords_morph_out_path + "xmm_peak_coords.csv")

### LoVoCCS-BCG UV luminosities

Luminosities calculated from GALEX cross-matches to the primary BCG candidate (i.e. BCG1) are read in:

In [5]:
lovoccs_bcg_uvlum = pd.read_csv("../../../LoVoCCS-BCG-Ident/outputs/vlass_galex_luminosities/BCG1_galex_luminosities.csv")
lovoccs_bcg_uvlum = lovoccs_bcg_uvlum.rename(columns={'cluster_name': 'name'})

# THIS SELECTS ONLY THE LUMINOSITY FOR THE GALEX MATCH WITH THE SMALLEST SEPARATION TO THE BCG - some cluster BCGs have 
#  multiple entries
sel_ind = lovoccs_bcg_uvlum.groupby('name').idxmin()['arcsec_sep_from_bcg']
lovoccs_bcg_uvlum = lovoccs_bcg_uvlum.loc[sel_ind, ['name', 'FUV_lum', 'err_FUV_lum', 'NUV_lum', 'err_NUV_lum']]

### LoVoCCS-BCG 3 GHz luminosities

Luminosities calculated from VLASS cross-matches to the primary BCG candidate (i.e. BCG1) are read in:

In [6]:
lovoccs_bcg_3GHzlum = pd.read_csv("../../../LoVoCCS-BCG-Ident/outputs/vlass_galex_luminosities/BCG1_vlass_luminosities.csv")
lovoccs_bcg_3GHzlum = lovoccs_bcg_3GHzlum.rename(columns={'cluster_name': 'name'})

# THIS SELECTS ONLY THE LUMINOSITY FOR THE VLASS MATCH WITH THE SMALLEST SEPARATION TO THE BCG - some cluster BCGs have 
#  multiple entries
sel_ind = lovoccs_bcg_3GHzlum.groupby('name').idxmin()['arcsec_sep_from_bcg']
lovoccs_bcg_3GHzlum = lovoccs_bcg_3GHzlum.loc[sel_ind, ['name', 'VLASS_lum', 'err_VLASS_lum']]

### Combining tables

In [7]:
xlovoccs_samp = pd.merge(xlovoccs_base, xlovoccs_peaks, left_on='name', right_on='name', how='outer')
xlovoccs_samp = pd.merge(xlovoccs_samp, lovoccs_bcg_uvlum, left_on='name', right_on='name', how='outer')
xlovoccs_samp = pd.merge(xlovoccs_samp, lovoccs_bcg_3GHzlum, left_on='name', right_on='name', how='outer')
xlovoccs_samp = xlovoccs_samp.sort_values('parent_LoVoCCSID').reset_index(drop=True)
xlovoccs_samp.head(6)

,name,LoVoCCSID,parent_LoVoCCSID,Name,start_ra,start_dec,MCXC_Redshift,MCXC_R500,MCXC_RA,MCXC_DEC,manual_xray_ra,manual_xray_dec,MCXC_Lx500_0.1_2.4,peak_ra,peak_dec,FUV_lum,err_FUV_lum,NUV_lum,err_NUV_lum,VLASS_lum,err_VLASS_lum
0,LoVoCCS-1,1,1,A2029,227.734300,5.745471,0.0766,1.3344,227.73000,5.720000,227.734300,5.745471,8.726709e+44,227.733253,5.745184,5.636077e+42,1.820455e+42,7.702808e+42,1.215087e+42,1.998871e+37,2.529629e+35
1,LoVoCCS-2,2,2,A401,44.740000,13.580000,0.0739,1.2421,44.74000,13.580000,NaN,NaN,6.088643e+44,44.750675,13.594567,NaN,NaN,2.312434e+42,7.735109e+41,NaN,NaN
2,LoVoCCS-4B,4B,4,A85South,10.451487,-9.460007,0.0555,1.2103,10.45875,-9.301944,10.451487,-9.460007,5.100085e+44,10.460369,-9.303375,1.306912e+42,4.140038e+41,2.297533e+42,4.370004e+41,1.378234e+36,5.714253e+34
3,LoVoCCS-4A,4A,4,A85North,10.458750,-9.301944,0.0555,1.2103,10.45875,-9.301944,NaN,NaN,5.100085e+44,10.460369,-9.303375,3.125557e+42,6.492113e+41,1.214686e+43,7.584305e+41,3.493556e+36,6.151529e+34
4,LoVoCCS-5,5,5,A3667,303.157313,-56.845978,0.0556,1.1990,303.13000,-56.830000,303.157313,-56.845978,4.871933e+44,303.100675,-56.851435,NaN,NaN,NaN,NaN,NaN,NaN
5,LoVoCCS-7,7,7,A3827,330.480000,-59.950000,0.0980,1.1367,330.48000,-59.950000,NaN,NaN,4.204419e+44,330.466217,-59.948751,7.534600e+42,1.649967e+42,2.359433e+43,1.492955e+42,NaN,NaN


## Defining an XGA ClusterSample

As we do not have observation cleaning turned on, we manually remove clusters that we know are excluded from our other analyses due to inadequate XMM data:

* LoVoCCS-41C is a component of 41 we identified from ROSAT Pointed data, but it does not fall on the XMM observation.
* LoVoCCS-33 is a cluster that partially falls on the edge of an observation of a nearby object - the coverage is insufficient for any real analysis however.

In [6]:
xlovoccs_samp = xlovoccs_samp[~xlovoccs_samp['LoVoCCSID'].isin(['41C', '33'])]

We define a ClusterSample, centered on the 'start positions' we defined in the early stages of our analysis of this sample - the start positions will be the same as the MCXC positions in cases where we judged the coordinate to be adequate, but will be manually defined from modern observations if it was too far outside the main part of the ICM, or if the cluster has multiple components that went unresolved in MCXC:

In [7]:
srcs = ClusterSample(xlovoccs_samp['start_ra'].values, xlovoccs_samp['start_dec'].values, xlovoccs_samp['MCXC_Redshift'].values, 
                     xlovoccs_samp['LoVoCCS_name'].values, r500=Quantity(xlovoccs_samp['MCXC_R500'].values, 'Mpc'), use_peak=False, 
                     clean_obs=False, cosmology=lovoccs_cosmo)
srcs.info()

Setting up Galaxy Clusters: 100%|██████████| 62/62 [04:03<00:00,  3.93s/it]


-----------------------------------------------------
Number of Sources - 62
Redshift Information - True
Sources with ≥1 detection - 60 [97%]
-----------------------------------------------------




/tmp/ipykernel_2297476/1780493561.py:1: UserWarning: The following do not appear to have any XMM data, and will not be included in the sample (can also check .failed_names); LoVoCCS-55, LoVoCCS-108, LoVoCCS-122
  srcs = ClusterSample(samp['start_ra'].values, samp['start_dec'].values, samp['MCXC_Redshift'].values,
/mnt/home/turne540/software/anaconda3/envs/xga_release/lib/python3.12/site-packages/xga/samples/extended.py:308: UserWarning: Non-fatal warnings occurred during the declaration of some sources, to access them please use the suppressed_warnings property of this sample.
  self._check_source_warnings()
